# KYC SUPERHERO WORKSHOP - STEP 4B: CROSS-REF & EDD CHECKER

The CHECKER layer for entity resolution and EDD assessment:

**4B.1: Validate Entity Cross-Reference**
- Are matches genuine (not false positives)?
- Are there missed matches (false negatives)?
- Is the EDD trigger decision appropriate?

**4B.2: Senior Compliance Review of EDD**
- Independent "senior officer" AI challenges the maker's assessment
- Compares risk scores for variance
- Checks regulatory completeness
- Provides APPROVED / REJECTED / ESCALATE decision

In [ ]:
%%sql -r context_setup
USE ROLE KYC_WORKSHOP_ROLE;
USE DATABASE KYC_SUPERHERO_DB;
USE SCHEMA CURATED;
USE WAREHOUSE KYC_WORKSHOP_WH;

In [ ]:
%%sql -r create_checked_crossref
CREATE OR REPLACE TABLE CHECKED_CROSSREF AS
SELECT
  x.file_name,
  x.applicant_name,
  x.HERO_NAME,
  x.REAL_NAME,
  x.known_risk_level,
  x.watchlist_match,
  x.watchlist_entity,
  x.watchlist_reason,
  x.edd_required,
  x.edd_trigger_reason,
  AI_COMPLETE(
    model => 'claude-sonnet-4-6',
    prompt => CONCAT(
      'You are a SENIOR KYC ANALYST validating entity matching results.\n',
      'Your job is to check whether the automated name-screening produced correct results.\n\n',
      '=== MATCHING RESULT TO VALIDATE ===\n',
      'Applicant Name (from KYC form): "', x.applicant_name, '"\n',
      'Matched to Known Entity: "', COALESCE(x.HERO_NAME, 'NO MATCH'), '"',
      CASE WHEN x.REAL_NAME IS NOT NULL THEN ' (Real name: "' || x.REAL_NAME || '")' ELSE '' END, '\n',
      'Match Confidence: ', CASE WHEN x.HERO_ID IS NOT NULL THEN 'POSITIVE' ELSE 'NO MATCH' END, '\n',
      'Watchlist Match: ', x.watchlist_match::VARCHAR, '\n',
      'Watchlist Entity: "', COALESCE(x.watchlist_entity, 'N/A'), '"\n',
      'EDD Required (system decision): ', x.edd_required::VARCHAR, '\n',
      'EDD Trigger Reason: ', x.edd_trigger_reason, '\n\n',
      '=== KNOWN ALIASES IN OUR DATABASE ===\n',
      '(Use these to check for false negatives)\n',
      '- Tony Stark = Iron Man = "Merchant of Death"\n',
      '- Steve Rogers = Captain America = "The First Avenger"\n',
      '- Natasha Romanoff = Black Widow = Natalia Alianovna Romanova\n',
      '- Bruce Banner = Hulk = "The Green Goliath" = "Joe Fixit"\n',
      '- Thor Odinson = "God of Thunder"\n',
      '- Peter Parker = Spider-Man = "The Web-Slinger"\n',
      '- T''Challa = Black Panther = "King of Wakanda"\n',
      '- Wanda Maximoff = Scarlet Witch\n',
      '- Stephen Strange = Doctor Strange = "Sorcerer Supreme"\n',
      '- Clint Barton = Hawkeye = "The Marksman"\n\n',
      '=== YOUR VALIDATION TASKS ===\n',
      '1. Is the hero match CORRECT? (Could the applicant name genuinely be this person?)\n',
      '2. Is the watchlist match GENUINE? (Or is it a coincidental/partial string match?)\n',
      '3. Are there MISSED matches? (Could this applicant be someone else in our database?)\n',
      '4. Is the EDD decision APPROPRIATE? Should it be upgraded, downgraded, or kept?\n',
      '5. Rate your confidence in the overall screening result.\n\n',
      'Remember: A false negative (missed match) is far more dangerous than a false positive.'
    ),
    response_format => {
      'type': 'json',
      'schema': {
        'type': 'object',
        'properties': {
          'check_status':           {'type': 'string'},
          'confidence_score':       {'type': 'integer'},
          'hero_match_valid':       {'type': 'boolean'},
          'watchlist_match_valid':  {'type': 'boolean'},
          'potential_missed_matches': {'type': 'array', 'items': {'type': 'string'}},
          'edd_decision_appropriate': {'type': 'boolean'},
          'recommended_action':     {'type': 'string'},
          'false_positive_risk':    {'type': 'string'},
          'false_negative_risk':    {'type': 'string'},
          'checker_notes':          {'type': 'string'}
        },
        'required': ['check_status', 'confidence_score', 'hero_match_valid', 
                     'watchlist_match_valid', 'edd_decision_appropriate', 'checker_notes']
      }
    }
  ) AS checker_output,
  CURRENT_TIMESTAMP() AS checked_at
FROM ENTITY_CROSSREF x;

In [ ]:
%%sql -r view_checked_crossref
SELECT
  file_name,
  applicant_name,
  COALESCE(HERO_NAME, '-- No Match --') AS matched_hero,
  checker_output:check_status::VARCHAR AS check_status,
  checker_output:confidence_score::INT AS confidence,
  checker_output:hero_match_valid::BOOLEAN AS match_valid,
  checker_output:edd_decision_appropriate::BOOLEAN AS edd_appropriate,
  checker_output:recommended_action::VARCHAR AS recommendation,
  checker_output:false_positive_risk::VARCHAR AS fp_risk,
  checker_output:false_negative_risk::VARCHAR AS fn_risk,
  checker_output:potential_missed_matches AS missed_matches,
  checker_output:checker_notes::VARCHAR AS notes
FROM CHECKED_CROSSREF
ORDER BY checker_output:confidence_score::INT ASC;

In [ ]:
%%sql -r create_checked_edd
CREATE OR REPLACE TABLE KYC_SUPERHERO_DB.ANALYTICS.CHECKED_EDD AS
SELECT
  e.file_name,
  e.applicant_name,
  e.known_risk_level,
  e.watchlist_match,
  e.watchlist_category,
  e.edd_trigger_reason,
  e.edd_assessment,
  e.applicant_occupation,
  e.source_of_funds,
  e.declared_pep_status,
  AI_COMPLETE(
    model => 'claude-sonnet-4-6',
    prompt => CONCAT(
      'You are a SENIOR COMPLIANCE OFFICER performing a SECOND-LINE REVIEW.\n',
      'Your role is to CHALLENGE the initial assessment, not rubber-stamp it.\n',
      'You report to the Head of Financial Crime and must ensure regulatory standards.\n\n',
      '=== APPLICANT ===\n',
      'Name: ', e.applicant_name, '\n',
      'Risk Level: ', COALESCE(e.known_risk_level, 'UNKNOWN'), '\n',
      'Watchlist Match: ', e.watchlist_match::VARCHAR, '\n',
      'Watchlist Category: ', COALESCE(e.watchlist_category, 'N/A'), '\n',
      'Watchlist Reason: ', COALESCE(e.watchlist_reason, 'N/A'), '\n',
      'EDD Trigger: ', e.edd_trigger_reason, '\n',
      'Occupation: ', COALESCE(e.applicant_occupation, 'Not stated'), '\n',
      'Source of Funds: ', COALESCE(e.source_of_funds, 'Not stated'), '\n',
      'PEP Status: ', COALESCE(e.declared_pep_status, 'Not declared'), '\n\n',
      '=== INITIAL (MAKER) EDD ASSESSMENT ===\n',
      e.edd_assessment::VARCHAR, '\n\n',
      '=== YOUR SENIOR REVIEW ===\n',
      'As the reviewing officer, evaluate:\n',
      '1. RISK SCORE: Is it appropriate? (Too lenient? Too harsh? Provide your own score)\n',
      '2. RISK FACTORS: Are all relevant factors identified? What''s missing?\n',
      '3. MITIGANTS: Are claimed mitigating factors legitimate?\n',
      '4. ACTIONS: Are recommended actions proportionate and complete?\n',
      '5. REGULATORY: Does this meet FCA/PRA expectations for EDD?\n',
      '   - Is source of wealth adequately investigated?\n',
      '   - Is ongoing monitoring proposed?\n',
      '   - Are reporting obligations (SAR/STR) considered?\n',
      '6. DECISION: APPROVED / REJECTED / ESCALATE_TO_MLRO\n\n',
      'Respond in JSON. Keep all values concise.\n',
      '=== CRITICAL OUTPUT CONSTRAINTS ===\n',
      'You MUST keep your response CONCISE to avoid truncation:\n',
      '- regulatory_compliance: MAX 2 sentences summarising compliance status.\n',
      '- missing_risk_factors: MAX 5 items, each MAX 20 words.\n',
      '- additional_actions_required: MAX 5 items, each MAX 15 words.\n',
      '- escalation_reason: MAX 2 sentences.\n',
      '- checker_notes: MAX 3 sentences.\n',
      '- All string fields must be brief and factual. Do NOT write paragraphs.'
    ),
    model_parameters => {'temperature': 0},
    response_format => {
      'type': 'json',
      'schema': {
        'type': 'object',
        'properties': {
          'check_status':               {'type': 'string'},
          'senior_risk_score':          {'type': 'integer'},
          'maker_risk_score_appropriate': {'type': 'boolean'},
          'score_variance':             {'type': 'integer'},
          'missing_risk_factors':       {'type': 'array', 'items': {'type': 'string'}},
          'additional_actions_required': {'type': 'array', 'items': {'type': 'string'}},
          'regulatory_compliance':      {'type': 'string'},
          'sar_consideration':          {'type': 'boolean'},
          'ongoing_monitoring_level':   {'type': 'string'},
          'escalation_reason':          {'type': 'string'},
          'final_recommendation':       {'type': 'string'},
          'checker_notes':              {'type': 'string'}
        },
        'required': ['check_status', 'senior_risk_score', 'maker_risk_score_appropriate',
                     'score_variance', 'missing_risk_factors', 'additional_actions_required',
                     'regulatory_compliance', 'sar_consideration', 'ongoing_monitoring_level',
                     'escalation_reason', 'final_recommendation', 'checker_notes'],
        'additionalProperties': false
      }
    }
  ) AS senior_review,
  CURRENT_TIMESTAMP() AS reviewed_at
FROM KYC_SUPERHERO_DB.ANALYTICS.EDD_ASSESSMENT e;

In [ ]:
%%sql -r view_checked_edd
SELECT
  e.applicant_name,
  e.edd_assessment:risk_score::INT AS maker_score,
  c.senior_review:senior_risk_score::INT AS checker_score,
  c.senior_review:score_variance::INT AS variance,
  c.senior_review:maker_risk_score_appropriate::BOOLEAN AS score_appropriate,
  e.edd_assessment:recommendation::VARCHAR AS maker_recommendation,
  c.senior_review:final_recommendation::VARCHAR AS checker_recommendation,
  c.senior_review:sar_consideration::BOOLEAN AS sar_needed,
  c.senior_review:ongoing_monitoring_level::VARCHAR AS monitoring_level,
  c.senior_review:checker_notes::VARCHAR AS senior_notes
FROM KYC_SUPERHERO_DB.ANALYTICS.CHECKED_EDD c
JOIN KYC_SUPERHERO_DB.ANALYTICS.EDD_ASSESSMENT e
  ON c.file_name = e.file_name
ORDER BY c.senior_review:senior_risk_score::INT DESC;

In [ ]:
%%sql -r checker_summary
SELECT
  'Cross-Reference' AS check_type,
  COUNT(*) AS total_checked,
  SUM(CASE WHEN checker_output:check_status::VARCHAR = 'PASS' THEN 1 ELSE 0 END) AS passed,
  SUM(CASE WHEN checker_output:check_status::VARCHAR = 'FAIL' THEN 1 ELSE 0 END) AS failed,
  SUM(CASE WHEN checker_output:check_status::VARCHAR = 'NEEDS_REVIEW' THEN 1 ELSE 0 END) AS needs_review,
  ROUND(AVG(checker_output:confidence_score::INT), 1) AS avg_confidence
FROM CHECKED_CROSSREF

UNION ALL

SELECT
  'EDD Assessment' AS check_type,
  COUNT(*) AS total_checked,
  SUM(CASE WHEN senior_review:check_status::VARCHAR ILIKE '%APPROVED%' THEN 1 ELSE 0 END) AS passed,
  SUM(CASE WHEN senior_review:check_status::VARCHAR ILIKE '%REJECTED%' THEN 1 ELSE 0 END) AS failed,
  SUM(CASE WHEN senior_review:check_status::VARCHAR ILIKE '%ESCALAT%' THEN 1 ELSE 0 END) AS needs_review,
  ROUND(100 - AVG(ABS(senior_review:score_variance::INT)) * 10, 1) AS avg_confidence
FROM KYC_SUPERHERO_DB.ANALYTICS.CHECKED_EDD;